<a href="https://colab.research.google.com/github/Raquelodaka/Teste-Hipotese-Email-Whatsapp/blob/main/Teste_de_Hipotese.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Resumo dos Dados:
- Total de Clientes: 1.000 apólices prestes a vencer
- Controle (E-mail): 503 clientes $\rightarrow$ 176 renovaram (35,0%)
- Tratamento (WhatsApp): 497 clientes $\rightarrow$ 231 renovaram (46,5%)

Antes de tocar em qualquer cálculo ou fórmula, precisamos definir o contexto do CRM da nossa seguradora.

O Cenário: Temos uma carteira de clientes com apólices de seguro auto/vida prestes a vencer.

O Dilema: Enviar e-mail é mais barato, mas pouca gente abre. Enviar WhatsApp gera mais engajamento, mas pode ter um custo por mensagem maior ou ser percebido como invasivo.

A Pergunta de Negócio: Mudar o canal de contato de E-mail para WhatsApp realmente melhora a taxa de renovação das apólices?

Na estatística, não podemos apenas olhar as taxas brutas de conversão e sair tirando conclusões, pois uma variação pequena pode acontecer por puro acaso (flutuação amostral). É para isso que serve o Teste de Hipóteses: ele nos dá um critério matemático para dizer se a diferença é real ou sorte.

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("/content/base_crm_renovacao_seguros.csv")
df

,id_cliente,grupo_experimento,idade,valor_premio_anual,renovou
0,CLI-1000,Controle (Email),43,1656.21,0
1,CLI-1001,Tratamento (WhatsApp),28,2450.14,1
2,CLI-1002,Tratamento (WhatsApp),45,1597.17,0
3,CLI-1003,Tratamento (WhatsApp),48,2956.03,0
4,CLI-1004,Controle (Email),47,2549.46,0
...,...,...,...,...,...
995,CLI-1995,Controle (Email),28,2329.07,0
996,CLI-1996,Tratamento (WhatsApp),35,2346.29,1
997,CLI-1997,Controle (Email),41,2355.10,1
998,CLI-1998,Tratamento (WhatsApp),37,2462.94,1


Realizaremos o calculo para verificar a taxa de renovação (proporção de conversão) de cada um dos dois grupos e, em seguida, subtraímos a taxa do Grupop "Tratamento (Whatsapp)" e "Controle(Email)"

In [ ]:
import pandas as pd

# 1. Agrupar por grupo e calcular o total (n), a soma de renovados (x) e a taxa média
resumo = df.groupby('grupo_experimento')['renovou'].agg(
    total_clientes='count',
    renovaram='sum',
    taxa_renovacao='mean'
)

# 2. Converter a taxa para percentual (%)
resumo['taxa_percentual'] = resumo['taxa_renovacao'] * 100

print(resumo)

# 3. Pegar os valores individuais
p_email = resumo.loc['Controle (Email)', 'taxa_percentual']
p_whatsapp = resumo.loc['Tratamento (WhatsApp)', 'taxa_percentual']

# 4. Calcular a diferença absoluta
diferenca = p_whatsapp - p_email
print(f"\nDiferença absoluta: {diferenca:.2f}%")

                       total_clientes  renovaram  taxa_renovacao  \
grupo_experimento                                                  
Controle (Email)                  503        176        0.349901   
Tratamento (WhatsApp)             497        231        0.464789   

                       taxa_percentual  
grupo_experimento                       
Controle (Email)             34.990060  
Tratamento (WhatsApp)        46.478873  

Diferença absoluta: 11.49%


Check de Randomização (ou Sanity Check)

Antes de testar se o WhatsApp performou melhor que o E-mail, precisamos garantir que o experimento foi limpo. Ou seja, devemos verificar se os dois grupos foram sorteados de forma homogênea e não possuem vieses que possam distorcer o resultado (como um grupo ter clientes mais novos ou seguros mais baratos que o outro).

Em testes A/B, as covariáveis (características dos usuários que já existiam antes do experimento, como idade e valor_premio_anual) devem estar equilibradas entre os grupos.Se houver diferença estatística nas características: Significa que o sorteio falhou (vício de seleção) e a diferença na taxa de renovação pode ser causada pela idade ou pelo preço, não pelo WhatsApp.Se não houver diferença estatística: Significa que a divisão foi justa e podemos atribuir o efeito observado diretamente ao canal de comunicação.Para testar isso em variáveis contínuas (idade e valor_premio_anual), usamos o Teste $t$ de Student para Duas Amostras Independentes (Teste $t$ de Welch).Hipóteses do Sanity Check:$H_0$ (Hipótese Nula): A média do grupo E-mail é igual à média do grupo WhatsApp ($\mu_{\text{Email}} = \mu_{\text{WhatsApp}}$).$H_1$ (Hipótese Alternativa): As médias são diferentes ($\mu_{\text{Email}} \neq \mu_{\text{WhatsApp}}$).Queremos que o $p$-valor seja maior que $0{,}05$ (não rejeitar $H_0$), o que confirma o equilíbrio dos grupos.

In [ ]:
import pandas as pd
import scipy.stats as stats

# Filtrar os dois grupos
g_email = df[df['grupo_experimento'] == 'Controle (Email)']
g_wapp = df[df['grupo_experimento'] == 'Tratamento (WhatsApp)']

# Variáveis que queremos testar
variaveis = ['idade', 'valor_premio_anual']

print("--- VERIFICAÇÃO DE RANDOMIZAÇÃO (SANITY CHECK) ---\n")

for var in variaveis:
    media_email = g_email[var].mean()
    media_wapp = g_wapp[var].mean()

    # Teste t de Welch (equal_var=False não assume variâncias iguais)
    t_stat, p_val = stats.ttest_ind(g_email[var], g_wapp[var], equal_var=False)

    print(f"Variável: {var}")
    print(f"  Média Controle (Email):     {media_email:.2f}")
    print(f"  Média Tratamento (WhatsApp): {media_wapp:.2f}")
    print(f"  Estatística t: {t_stat:.4f} | p-valor: {p_val:.4f}")

    if p_val > 0.05:
        print("  Conclusão: Grupos balanceados (Sem diferença estatística).\n")
    else:
        print("  Conclusão: ATENÇÃO! Grupos desbalanceados.\n")

--- VERIFICAÇÃO DE RANDOMIZAÇÃO (SANITY CHECK) ---

Variável: idade
  Média Controle (Email):     42.71
  Média Tratamento (WhatsApp): 42.41
  Estatística t: 0.4863 | p-valor: 0.6268
  Conclusão: Grupos balanceados (Sem diferença estatística).

Variável: valor_premio_anual
  Média Controle (Email):     2485.50
  Média Tratamento (WhatsApp): 2531.36
  Estatística t: -1.2461 | p-valor: 0.2130
  Conclusão: Grupos balanceados (Sem diferença estatística).



In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats

# 2. Contagens de cada grupo
n_wapp = len(df[df['grupo_experimento'] == 'Tratamento (WhatsApp)'])
x_wapp = df[df['grupo_experimento'] == 'Tratamento (WhatsApp)']['renovou'].sum()

n_email = len(df[df['grupo_experimento'] == 'Controle (Email)'])
x_email = df[df['grupo_experimento'] == 'Controle (Email)']['renovou'].sum()

# 3. Proporções individuais e diferença (SINAL)
p_wapp = x_wapp / n_wapp
p_email = x_email / n_email
diferenca = p_wapp - p_email  # 0.114888 (11.49%)

# 4. Proporção Agrupada (p_pool)
p_pool = (x_wapp + x_email) / (n_wapp + n_email)  # 0.407 (40.7%)

# 5. Erro Padrão (RUÍDO DA AMOSTRA) -> Aqui estão os 3,11%!
erro_padrao = np.sqrt(p_pool * (1 - p_pool) * ((1 / n_wapp) + (1 / n_email)))

# 6. Estatística Z (Sinal / Ruído)
z_calculado = diferenca / erro_padrao

# 7. p-valor a partir da Distribuição Normal
p_valor_calculado = 2 * (1 - stats.norm.cdf(abs(z_calculado)))

# Exibir os resultados passo a passo
print(f"Diferença (Sinal):          {diferenca * 100:.2f}%")
print(f"Erro Padrão (Ruído):        {erro_padrao * 100:.2f}%")
print(f"Estatística Z (Manual):     {z_calculado:.4f}")
print(f"p-valor (Manual):           {p_valor_calculado:.6f}")

Diferença (Sinal):          11.49%
Erro Padrão (Ruído):        3.11%
Estatística Z (Manual):     3.6975
p-valor (Manual):           0.000218


### 📊 Entendendo o Teste Z e o p-valor no Nosso Experimento

Para validar se o aumento na taxa de renovação do **WhatsApp (46,48%)** em relação ao **E-mail (34,99%)** é um ganho real e não apenas "sorte do sorteio", aplicamos o **Teste Z para Duas Proporções**.

---

#### 1. O que é o Teste Z neste caso?
* **O Conceito:** O Teste Z mede o **"sinal"** (a diferença de $11,49\%$ entre os grupos) em relação ao **"ruído"** (a oscilação/variação natural esperada em amostras desse tamanho, que calculamos em $\approx 3,11\%$).
* **A Fórmula Prática:**
  $$\text{Estatística } Z = \frac{\text{Diferença Observada}}{\text{Ruído da Amostra (Erro Padrão)}} = \frac{11,49\%}{3,11\%} = \mathbf{3,70}$$
* **O Resultado:** Obter um $Z = 3,70$ significa que o resultado do WhatsApp ficou **3,7 vezes mais distante do zero** do que qualquer oscilação normal causada pelo acaso.

---

#### 2. O que é o p-valor neste caso?
* **O Conceito:** O $p$-valor pega a estatística $Z$ e a transforma em uma probabilidade ($0\%$ a $100\%$). Ele responde à pergunta: *"Se o WhatsApp e o E-mail fossem rigorosamente iguais, qual seria a chance de ver uma diferença de $11,49\%$ por mero acaso?"*
* **O Resultado:** O nosso $p$-valor foi de **$0,000218$** (ou **$0,02\%$**).

---

#### 3. Regra de Decisão Estatística
* **Critério de Corte ($\alpha$):** Adotamos o padrão do mercado de **$5\%$ ($0,05$)**.
* **Comparação:** Como nosso $p\text{-valor } (0,02\%) \le 5\%$, **rejeitamos a Hipótese Nula ($H_0$)**.

> **Conclusão para o Negócio:** A probabilidade dessa melhoria ser apenas coincidência é de apenas $1$ em $5.000$ casos ($0,02\%$). Portanto, temos segurança estatística ($99,98\%$ de confiança) de que o **WhatsApp é significativamente superior ao E-mail** na renovação de seguros.

In [ ]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

# 2. Agrupar sucessos (x) e totais (n)
resumo = df.groupby('grupo_experimento')['renovou'].agg(['sum', 'count'])

sucessos = [
    resumo.loc['Tratamento (WhatsApp)', 'sum'],
    resumo.loc['Controle (Email)', 'sum']
]

amostras = [
    resumo.loc['Tratamento (WhatsApp)', 'count'],
    resumo.loc['Controle (Email)', 'count']
]

# 3. Executar o Teste Z para duas proporções
z_stat, p_valor = proportions_ztest(count=sucessos, nobs=amostras, alternative='two-sided')

print(f"Estatística Z: {z_stat:.4f}")
print(f"p-valor: {p_valor:.6f}")

# 4. Decisão Estatística
alpha = 0.05
if p_valor < alpha:
    print("\nConclusão: Rejeitamos a Hipótese Nula (H0).")
    print("A diferença de conversão a favor do WhatsApp É estatisticamente significante!")
else:
    print("\nConclusão: Não Rejeitamos a Hipótese Nula (H0).")
    print("Não há evidências suficientes de que um canal seja melhor que o outro.")

Estatística Z: 3.6975
p-valor: 0.000218

Conclusão: Rejeitamos a Hipótese Nula (H0).
A diferença de conversão a favor do WhatsApp É estatisticamente significante!


### 📏 Intervalo de Confiança (95%) para a Diferença de Proporções

Enquanto o Teste Z confirma que o WhatsApp é estatisticamente superior ao E-mail, o **Intervalo de Confiança (IC)** define a **margem de ganho esperada** no mundo real.

---

#### 1. O que este cálculo faz?
* **Diferença Observada ($11,49\%$):** É o resultado obtido especificamente na nossa amostra de $1.000$ clientes.
* **Margem de Erro ($\pm 6,05\%$):** É a variação esperada baseada em um nível de confiança de $95\%$ ($Z_{\text{crítico}} = 1,96$).
* **Intervalo Resultante ($[5,44\%, 17,54\%]$):** É a faixa realista onde a verdadeira diferença populacional se encontra.

---

#### 2. Como interpretar o resultado no Notebook?
> **Interpretação Estatística:** Temos $95\%$ de confiança de que, ao aplicar a comunicação via WhatsApp para toda a base de clientes da empresa, a taxa de renovação aumentará entre **$+5,44\%$** (cenário conservador) e **$+17,54\%$** (cenário otimista) em comparação ao E-mail.

---

#### 3. Aplicação Prática para o Negócio
* **Projeção de Cenário Mínimo:** Mesmo no pior cenário dentro dos $95\%$ de confiança ($+5,44\%$), a mudança de canal traz um ganho líquido relevante de retenção sem custos proporcionais de infraestrutura.
* **Decisão:** O intervalo **não cruza o valor zero** (ambos os limites são positivos), reafirmando que a migração para o WhatsApp é uma decisão de baixo risco e alto retorno.

In [ ]:
import numpy as np
import scipy.stats as stats

# 5. Calcular o Intervalo de Confiança de 95% para a Diferença
p_wapp = sucessos[0] / amostras[0]
p_email = sucessos[1] / amostras[1]
diferenca = p_wapp - p_email

# Erro padrão da diferença entre duas proporções independentes
se_diff = np.sqrt((p_wapp * (1 - p_wapp) / amostras[0]) + (p_email * (1 - p_email) / amostras[1]))

# Margem de erro (Z_crítico = 1.96 para 95% de confiança)
z_critico = stats.norm.ppf(0.975)
margem_erro = z_critico * se_diff

ic_inferior = (diferenca - margem_erro) * 100
ic_superior = (diferenca + margem_erro) * 100

print(f"\nDiferença Pontual Observada: {diferenca * 100:.2f}%")
print(f"Margem de Erro: ±{margem_erro * 100:.2f}%")
print(f"Intervalo de Confiança (95%): [{ic_inferior:.2f}%, {ic_superior:.2f}%]")


Diferença Pontual Observada: 11.49%
Margem de Erro: ±6.05%
Intervalo de Confiança (95%): [5.44%, 17.54%]


### 🔎 Análise Segmentada: Onde o WhatsApp Gera Maior Valor?

A análise de subgrupos permite identificar se o impacto do WhatsApp é homogêneo em toda a base ou se existem perfis de clientes específicos que respondem melhor ao novo canal de comunicação.

---

#### 1. Principais Achados por Faixa Etária

| Faixa Etária | Taxa E-mail | Taxa WhatsApp | Ganho Incremental (p.p.) |
| :--- | :---: | :---: | :---: |
| **20 a 35 anos** | $39,67\%$ | $42,98\%$ | **$+3,31$ p.p.** |
| **36 a 50 anos** | $33,09\%$ | $48,55\%$ | **$+15,46$ p.p.** *(Maior Ganho)* |
| **51+ anos** | $34,51\%$ | $45,00\%$ | **$+10,49$ p.p.** |

* **Insight:** O público de **36 a 50 anos** apresentou a maior sensibilidade ao canal WhatsApp, saltando de $33,09\%$ para $48,55\%$ de conversão.
* **Público Jovem (20–35):** Apresenta conversão alta no E-mail em relação aos demais grupos, resultando em um ganho incremental menor via WhatsApp.

---

#### 2. Principais Achados por Faixa de Valor do Prêmio (Ticket)

| Faixa de Prêmio | Taxa E-mail | Taxa WhatsApp | Ganho Incremental (p.p.) |
| :--- | :---: | :---: | :---: |
| **Prêmio Baixo** | $36,99\%$ | $44,72\%$ | **$+7,73$ p.p.** |
| **Prêmio Médio** | $34,10\%$ | $45,63\%$ | **$+11,52$ p.p.** |
| **Prêmio Alto** | $33,76\%$ | $48,86\%$ | **$+15,11$ p.p.** *(Maior Ganho)* |

* **Insight:** Quanto **maior o valor da apólice**, maior é a eficiência do WhatsApp em comparação ao E-mail.
* **Efeito no Faturamento:** Apólices de valor elevado demandam uma comunicação mais agilizada e direta para tirar dúvidas de renovação, tornando o WhatsApp um canal estratégico para retenção de receitas altas.

---

#### 💡 Aplicação Prática para Estratégia de CRM
1. **Priorização de Disparos:** Caso haja limite de orçamento na API oficial do WhatsApp, a prioridade máxima de disparo deve ser para **clientes acima de 35 anos** e com **prêmios médios/altos**.
2. **Otimização de Custos:** Para o público jovem com apólices baratas, manter uma régua de comunicação mista (E-mail como canal primário e WhatsApp apenas de reforço/D-2) garante alto ROI sem desperdício de disparos pagos.

In [ ]:
# 6. Criar faixas para Idade e Valor do Prêmio
df['faixa_etaria'] = pd.cut(df['idade'], bins=[20, 35, 50, 75], labels=['20-35 anos', '36-50 anos', '51+ anos'])
df['faixa_premio'] = pd.qcut(df['valor_premio_anual'], q=3, labels=['Prêmio Baixo', 'Prêmio Médio', 'Prêmio Alto'])

print("\n--- TAXA DE RENOVAÇÃO (%) POR FAIXA ETÁRIA ---")
print(pd.crosstab(df['grupo_experimento'], df['faixa_etaria'], values=df['renovou'], aggfunc='mean') * 100)

print("\n--- TAXA DE RENOVAÇÃO (%) POR FAIXA DE PRÊMIO ---")
print(pd.crosstab(df['grupo_experimento'], df['faixa_premio'], values=df['renovou'], aggfunc='mean') * 100)


--- TAXA DE RENOVAÇÃO (%) POR FAIXA ETÁRIA ---
faixa_etaria           20-35 anos  36-50 anos   51+ anos
grupo_experimento                                       
Controle (Email)        39.669421   33.085502  34.513274
Tratamento (WhatsApp)   42.975207   48.550725  45.000000

--- TAXA DE RENOVAÇÃO (%) POR FAIXA DE PRÊMIO ---
faixa_premio           Prêmio Baixo  Prêmio Médio  Prêmio Alto
grupo_experimento                                             
Controle (Email)          36.994220     34.104046    33.757962
Tratamento (WhatsApp)     44.720497     45.625000    48.863636


## 💼 Relatório Executivo: Impacto do WhatsApp na Renovação de Seguros

### 1. Resumo Executivo
Realizamos um Teste A/B com **1.000 clientes** para avaliar se a alteração do canal de comunicação de renovação — trocando o **E-mail** pelo **WhatsApp** — traria ganho real na taxa de retenção da carteira. O canal WhatsApp apresentou um desempenho superior e validado estatisticamente.

---

### 2. Principais Resultados Obtidos
* **Aumento de Conversão**: O WhatsApp alcançou uma taxa de renovação de **46,5%**, contra **35,0%** do E-mail. Isso representa um ganho absoluto de **+11,5 pontos percentuais** (+32,8% de crescimento relativo).
* **Segurança da Decisão (Zero "Sorte")**: A probabilidade de esse resultado ter sido fruto do acaso é de apenas **0,02%** ($1$ em $5.000$ chances). A decisão de mudança tem **99,98% de confiança estatística**.
* **Projeção para a Base Total**: Ao expandir o WhatsApp para toda a base da empresa, estimamos um ganho contínuo de conversão entre **+5,4%** (cenário conservador) e **+17,5%** (cenário otimista).

---

### 3. Impacto Financeiro na Amostra
No grupo testado ($1.000$ clientes com prêmio médio de R$ 2.508,29):
* **Renovações via E-mail**: 176 apólices $\rightarrow$ **R$ 441.447,95** em receita.
* **Renovações via WhatsApp**: 231 apólices $\rightarrow$ **R$ 584.744,10** em receita.
* **Ganho Incremental Direto**: **+55 apólices renovadas** gerando **+R$ 143.296,15** adicionais em receita de prêmios durante o teste.

---

### 4. Onde Está a Maior Oportunidade? (Segmentação)
1. **Clientes de 36 a 50 anos**: Apresentaram a maior sensibilidade ao WhatsApp, aumentando a conversão em **+15,5 p.p.** (de 33,1% no E-mail para 48,5% no WhatsApp).
2. **Apólices de Valor Alto**: Clientes com seguros mais caros respondem melhor ao WhatsApp, com salto de **+15,1 p.p.** na taxa de renovação.

---

### 5. Recomendação Estratégica
Aprovar a migração imediata do E-mail para o **WhatsApp como canal primário de renovação** na régua de CRM, priorizando o envio para clientes acima de 35 anos e com apólices de maior valor para otimizar o retorno sobre o investimento.

## 🔬 Relatório Técnico: Teste de Hipótese para Duas Proporções

### 1. Metodologia e Desenho do Experimento
Trata-se de um ensaio controlado e aleatorizado (*Randomized Controlled Trial*) com amostra de $N = 1.000$ observações independentes para testar a taxa de conversão (`renovou` $\in \{0, 1\}$) entre dois tratamentos de CRM:
* **Grupo Controle (E-mail)**: $n_1 = 503$
* **Grupo Tratamento (WhatsApp)**: $n_2 = 497$

---

### 2. Formulação das Hipóteses e Parâmetros
* **Hipótese Nula ($H_0$)**: $p_{\text{WhatsApp}} - p_{\text{Email}} = 0$ *(Sem efeito de tratamento)*
* **Hipótese Alternativa ($H_1$)**: $p_{\text{WhatsApp}} - p_{\text{Email}} \neq 0$ *(Efeito bidirecional)*
* **Nível de Significância ($\alpha$)**: $0{,}05$ ($95\%$ de nível de confiança)

---

### 3. Sanity Check (Validação do Sorteio)
Aplicou-se o **Teste $t$ de Welch** para amostras independentes sobre as covariáveis pré-tratamento para descartar viés de seleção (*confounding bias*):
* **Idade**: $t = 0{,}4863$, $p\text{-valor} = 0{,}6268$
* **Valor do Prêmio Anual**: $t = -1{,}2461$, $p\text{-valor} = 0{,}2130$

Como $p > 0{,}05$ em ambos os casos, confirma-se o balanceamento de covariáveis e a integridade da randomização.

---

### 4. Estatística do Teste $Z$ de Proporções Agrupadas
* **Taxas Observadas**: $\hat{p}_{\text{Email}} = 34{,}99\%$ ($176/503$) e $\hat{p}_{\text{WhatsApp}} = 46{,}48\%$ ($231/497$).
* **Diferença Observada ($\Delta \hat{p}$)**: $+11{,}49\%$ ($0{,}11489$).
* **Proporção Agrupada ($\bar{p}$)**: $\frac{176 + 231}{503 + 497} = 0{,}4070$.
* **Erro Padrão Agrupado ($\text{SE}$)**:
  $$\text{SE} = \sqrt{0{,}4070 \times (1 - 0{,}4070) \times \left(\frac{1}{497} + \frac{1}{503}\right)} \approx 0{,}03107 \quad (3{,}11\%)$$
* **Estatística $Z$ Calculada**:
  $$Z = \frac{0{,}11489}{0{,}03107} = \mathbf{3{,}6975}$$
* **$p$-valor (bicaudal)**: $\mathbf{0{,}000218}$ ($2{,}18 \times 10^{-4}$).

---

### 5. Intervalo de Confiança da Diferença
Calculado via aproximação normal com erro padrão não-agrupado para diferença de duas proporções com $\alpha = 0{,}05$ ($Z_{\alpha/2} = 1{,}96$):
$$\text{IC}_{95\%} = \Delta \hat{p} \pm 1{,}96 \times \text{SE}_{\text{diff}} = [+\mathbf{5{,}44\%}, +\mathbf{17{,}54\%}]$$

---

### 6. Conclusão Estatística
Como $p\text{-valor} = 0{,}000218 < \alpha = 0{,}05$, **rejeita-se a hipótese nula ($H_0$)**. Há evidências estatísticas altamente consistentes para afirmar que a taxa de conversão do canal WhatsApp é superior à do E-mail nesta população.